# 04 · Governança e evidências (versão Spark)

Complemento dos notebooks `01_bronze`, `02_silver` e `03_gold` (Spark). Aqui ficam as partes da atividade que o `03_gold` Spark não cobria:

1. Inventário do storage (landing e warehouse) e contagem de linhas por tabela.
2. DQ como **histórico**: cada execução vira um snapshot novo em `lake.silver.dq_resultados`.
3. **Versionamento do bucket** (MinIO): o mesmo erro humano do `02_silver`, recuperado pelo object storage.
4. Gold com **contrato de dados**, **nome mascarado**, **gates** e **reconciliação**.
5. Publicação no Postgres com **TRUNCATE** e usuário **somente leitura** `metabase_ro` para o Metabase.

In [1]:
import os, sys, io, json
# Spark 3.5 precisa de Java 17 (o Java padrão desta máquina é o 25, que quebra o Spark)
_jdk17 = os.path.expanduser("~/.sdkman/candidates/java/17.0.14-tem")
if os.path.isdir(_jdk17):
    os.environ["JAVA_HOME"] = _jdk17
    os.environ["PATH"] = f"{_jdk17}/bin:" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable  # worker = mesmo Python do notebook
sys.path.insert(0, os.path.abspath(".."))
from common import lake_local as L, medalhao as M
from common.dq import DQ
from pyspark.sql import functions as F
import pandas as pd
spark = L.get_spark()
s3 = L._s3()
print("Spark", spark.version, "| master:", spark.sparkContext.master, "| storage:", L.STORAGE)

26/11/21 06:28:39 WARN Utils: Your hostname, ari-note resolves to a loopback address: 127.0.1.1; using 192.168.17.108 instead (on interface wlp44s0)
26/11/21 06:28:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ari/pos/wuldson/arqu_coleta_dados_pos_unipe/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ari/.ivy2/cache
The jars for the packages stored in: /home/ari/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-62a64a06-842c-4a34-9a01-11813e1e66bc;1.0
	confs: [default]


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central


	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 622ms :: artifacts dl 48ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   6   |   0   |   0  

:: retrieving :: org.apache.spark#spark-submit-parent-62a64a06-842c-4a34-9a01-11813e1e66bc
	confs: [default]
	0 artifacts copied, 6 already retrieved (0kB/22ms)


26/11/21 06:28:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | master: local[2] | storage: minio


## 1 · Inventário do storage
Landing = arquivos crus (CSV/JSONL). Warehouse = tabelas Iceberg (Parquet em `data/` + `metadata/`).

In [2]:
print("BUCKET landing")
for chave, tam in L.list_landing("mercadonordeste/"):
    print(f"  {tam:>10,} bytes  {chave}")

linhas = []
for chave, tam in L.list_warehouse("lakehouse/"):
    partes = chave.split("/")            # lakehouse/<camada>/<tabela>/(data|metadata)/...
    linhas.append({"camada": partes[1], "tabela": partes[2],
                   "tipo": partes[3] if len(partes) > 3 else "-", "bytes": tam})
inv = (pd.DataFrame(linhas).pivot_table(index=["camada", "tabela"], columns="tipo", values="bytes",
                                        aggfunc="count", fill_value=0)
       .rename(columns={"data": "objetos_data", "metadata": "objetos_metadata"}))
inv["KB"] = pd.DataFrame(linhas).groupby(["camada", "tabela"])["bytes"].sum().div(1024).round().astype(int)
print("\nBUCKET warehouse (prefixo lakehouse/)")
inv

BUCKET landing
     529,223 bytes  mercadonordeste/lote1/avaliacoes.csv
     420,891 bytes  mercadonordeste/lote1/clientes.csv
   7,571,082 bytes  mercadonordeste/lote1/pedidos.jsonl
       9,044 bytes  mercadonordeste/lote1/produtos.csv



BUCKET warehouse (prefixo lakehouse/)


tipo                      objetos_data  objetos_metadata    KB
camada tabela                                                 
bronze avaliacoes                    2                10   179
       clientes                      2                10   174
       pedidos                       2                10   972
       produtos                      2                10    57
gold   avaliacoes_resumo             1                 4    16
       clientes_360                  1                 4   103
       vendas_diarias                1                 4   182
silver avaliacoes                    1                 4    77
       clientes                      1                 4    80
       dq_resultados                18                22   168
       itens_pedido                  1                 4   313
       itens_pedido_aula             2                 9  1300
       pedidos                       1                 4   319
       produtos                      1                 4    20
       quarentena                    3                 4    17

## 2 · Contagem de linhas por tabela (todas as camadas)

In [3]:
cont = []
for ns in ("bronze", "silver", "gold"):
    for r in spark.sql(f"SHOW TABLES IN lake.{ns}").collect():
        t = f"lake.{ns}.{r.tableName}"
        cont.append({"camada": ns, "tabela": r.tableName, "linhas": spark.table(t).count()})
pd.DataFrame(cont).sort_values(["camada", "tabela"]).reset_index(drop=True)

,camada,tabela,linhas
0,bronze,avaliacoes,6489
1,bronze,clientes,5000
2,bronze,pedidos,30150
3,bronze,produtos,200
4,gold,avaliacoes_resumo,112
5,gold,clientes_360,5000
6,gold,vendas_diarias,30129
7,silver,avaliacoes,6464
8,silver,clientes,5000
9,silver,dq_resultados,24


## 3 · DQ como histórico
No `02_silver` o relatório foi gravado com `createOrReplace`. Aqui usamos `DQ.save()`, que faz **append**: cada execução vira um **snapshot novo** (e arquivos novos no storage), sem apagar a anterior. Assim dá para ver a qualidade **ao longo do tempo**.

In [4]:
cli = spark.table("lake.silver.clientes")
ped = spark.table("lake.silver.pedidos")
av  = spark.table("lake.silver.avaliacoes")
DQ(spark, cli, "silver.clientes").not_null("cliente_id").unique("cliente_id").not_null("email", "aviso").save()
DQ(spark, ped, "silver.pedidos").not_null("pedido_id").unique("pedido_id") \
    .in_set("status", ["pago", "enviado", "entregue", "cancelado", "devolvido"]).save()
DQ(spark, av, "silver.avaliacoes").unique("avaliacao_id").between("nota", 1, 5).save()

spark.sql("""SELECT substr(executado_em, 1, 16) AS execucao, count(*) AS regras,
                     sum(CASE WHEN status = 'OK' THEN 1 ELSE 0 END) AS ok,
                     sum(CASE WHEN status <> 'OK' THEN 1 ELSE 0 END) AS aviso_ou_erro
              FROM lake.silver.dq_resultados GROUP BY 1 ORDER BY 1""").show(truncate=False)
spark.sql("""SELECT committed_at, snapshot_id, operation, summary['added-records'] AS linhas_adicionadas,
                     summary['total-records'] AS total FROM lake.silver.dq_resultados.snapshots
              ORDER BY committed_at""").show(truncate=False)
print("objetos de dados do DQ no storage:",
      sum("/data/" in c for c, _ in L.list_warehouse("lakehouse/silver/dq_resultados/")))
spark.table("lake.silver.dq_resultados").filter("status <> 'OK'").show(truncate=False)

+----------------+------+---+-------------+
|execucao        |regras|ok |aviso_ou_erro|
+----------------+------+---+-------------+
|2026-11-05T22:26|8     |7  |1            |
|2026-11-21T06:26|8     |7  |1            |
|2026-11-21T06:28|8     |7  |1            |
|2026-11-21T06:29|8     |7  |1            |
+----------------+------+---+-------------+



+-----------------------+-------------------+---------+------------------+-----+
|committed_at           |snapshot_id        |operation|linhas_adicionadas|total|
+-----------------------+-------------------+---------+------------------+-----+
|2026-11-05 22:26:32.315|2414446842388077371|append   |8                 |8    |
|2026-11-21 06:26:40.462|5482900344969410665|append   |3                 |11   |
|2026-11-21 06:26:46.825|4142005914251285042|append   |3                 |14   |
|2026-11-21 06:26:50.268|5035747887925779056|append   |2                 |16   |
|2026-11-21 06:28:07.926|2855500362039233985|append   |3                 |19   |
|2026-11-21 06:28:10.75 |7844121739274637231|append   |3                 |22   |
|2026-11-21 06:28:12.414|1980089672687517089|append   |2                 |24   |
|2026-11-21 06:29:24.423|2616763591691873029|append   |3                 |27   |
|2026-11-21 06:29:28.515|7489216844527636390|append   |3                 |30   |
|2026-11-21 06:29:30.881|190

+---------------+--------+------+-----+------+---------+------+-------------------+
|dataset        |regra   |coluna|total|falhas|pct_falha|status|executado_em       |
+---------------+--------+------+-----+------+---------+------+-------------------+
|silver.clientes|not_null|email |5000 |270   |5.4      |AVISO |2026-11-21T06:29:21|
|silver.clientes|not_null|email |5000 |270   |5.4      |AVISO |2026-11-21T06:28:04|
|silver.clientes|not_null|email |5000 |270   |5.4      |AVISO |2026-11-05T22:26:17|
|silver.clientes|not_null|email |5000 |270   |5.4      |AVISO |2026-11-21T06:26:34|
+---------------+--------+------+-----+------+---------+------+-------------------+



## 4 · Versionamento do bucket (recurso do object storage)
No `02_silver` recuperamos o erro com o histórico do **Iceberg** (`VERSION AS OF` + `rollback_to_snapshot`). Agora o mesmo erro com o recurso do **próprio MinIO**: um bucket com versionamento ligado guarda todas as versões de cada objeto.

In [5]:
BUCKET_H, CHAVE_H = "historico", "silver/itens_pedido_aula.parquet"
if BUCKET_H not in [b["Name"] for b in s3.list_buckets()["Buckets"]]:
    s3.create_bucket(Bucket=BUCKET_H)
s3.put_bucket_versioning(Bucket=BUCKET_H, VersioningConfiguration={"Status": "Enabled"})
print("versionamento:", s3.get_bucket_versioning(Bucket=BUCKET_H)["Status"])

def gravar(df):
    buf = io.BytesIO(); df.to_parquet(buf, index=False)
    return s3.put_object(Bucket=BUCKET_H, Key=CHAVE_H, Body=buf.getvalue())["VersionId"]

def ler(version_id=None):
    extra = {"VersionId": version_id} if version_id else {}
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=BUCKET_H, Key=CHAVE_H, **extra)["Body"].read()))

_df = spark.table("lake.silver.itens_pedido_aula")
itens_pd = pd.DataFrame(_df.collect(), columns=_df.columns)   # toPandas() exige distutils (ausente no Python 3.12)
v_boa = gravar(itens_pd)                                             # versão correta
v_ruim = gravar(itens_pd[itens_pd["status"] != "cancelado"])         # o erro humano: apagar cancelados
print(f"atual (errada): {len(ler()):,} linhas | versão boa {v_boa[:8]}…: {len(ler(v_boa)):,} linhas")

vers = s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)["Versions"]
print(pd.DataFrame([{"version_id": v["VersionId"][:12] + "…", "gravado_em": v["LastModified"],
                     "bytes": v["Size"], "atual": v["IsLatest"]} for v in vers]).to_string(index=False))

# recuperar = copiar a versão boa por cima (vira uma 3ª versão; nada é apagado)
s3.copy_object(Bucket=BUCKET_H, Key=CHAVE_H, CopySource={"Bucket": BUCKET_H, "Key": CHAVE_H, "VersionId": v_boa})
print(f"\ndepois da recuperação: {len(ler()):,} linhas | versões guardadas:",
      len(s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)["Versions"]))

versionamento: Enabled


atual (errada): 43,886 linhas | versão boa dcb9cd2c…: 50,105 linhas
   version_id                       gravado_em  bytes  atual
e9aa4e82-93b… 2026-11-21 09:29:39.942000+00:00 799503   True
dcb9cd2c-a85… 2026-11-21 09:29:39.866000+00:00 916263  False

depois da recuperação: 50,105 linhas | versões guardadas: 3


In [6]:
# LGPD: com versionamento, DELETE só cria uma "marca de exclusão"; os dados antigos continuam lá
s3.delete_object(Bucket=BUCKET_H, Key=CHAVE_H)
r = s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)
print("versões com dado ainda guardadas:", len(r.get("Versions", [])), "| marcas de exclusão:", len(r.get("DeleteMarkers", [])))
# apagar de verdade = remover cada VersionId (é o que o direito ao esquecimento exige)
for v in r.get("Versions", []) + r.get("DeleteMarkers", []):
    s3.delete_object(Bucket=BUCKET_H, Key=CHAVE_H, VersionId=v["VersionId"])
r = s3.list_object_versions(Bucket=BUCKET_H, Prefix=CHAVE_H)
print("depois de apagar cada versão:", len(r.get("Versions", [])), "versões,", len(r.get("DeleteMarkers", [])), "marcas")

versões com dado ainda guardadas: 3 | marcas de exclusão: 1


depois de apagar cada versão: 0 versões, 0 marcas


## 5 · Gold: contrato de dados antes do código
O contrato diz **o que cada tabela é** (grão, dono, colunas). Ele fica no código porque os gates o usam e é publicado junto com a Gold.

In [7]:
CONTRATOS = {
  "vendas_diarias": {
    "grao": "data x categoria x estado x canal_aquisicao", "chave": ["data", "categoria", "estado", "canal_aquisicao"],
    "dono": "Comercial (exemplo)", "atualizacao": "reconstruída a cada execução", "pii": False,
    "descricao": "Vendas por dia. Exclui pedidos cancelados. Itens de produto em quarentena entram como categoria 'nao_classificado'.",
    "colunas": {"data": "Dia do pedido", "categoria": "Categoria do produto ('nao_classificado' se o produto está em quarentena)",
                "estado": "UF do cliente", "canal_aquisicao": "Canal de aquisição do cliente",
                "qtd_pedidos": "Pedidos distintos no grupo", "qtd_itens": "Soma de qtd dos itens",
                "receita": "Soma de valor_item (R$)"}},
  "clientes_360": {
    "grao": "um cliente", "chave": ["cliente_id"], "dono": "CRM (exemplo)",
    "atualizacao": "reconstruída a cada execução", "pii": True,
    "descricao": "Visão do cliente. Nome mascarado; e-mail não entra na Gold (minimização).",
    "colunas": {"cliente_id": "Identificador do cliente (pseudônimo)", "nome_mascarado": "Nome com letras ocultas (A** S****)",
                "estado": "UF", "canal_aquisicao": "Canal de aquisição", "data_cadastro": "Data de cadastro",
                "qtd_pedidos": "Pedidos não cancelados", "receita_total": "Receita do cliente (R$)",
                "ticket_medio": "Média por pedido (R$)", "ultima_compra": "Data da última compra",
                "nota_media": "Média das notas que deu", "recencia_dias": "Dias desde a última compra até o último dia da base",
                "segmento": "sem_compra | inativo | campeao | ativo | em_risco"}},
  "avaliacoes_resumo": {
    "grao": "mês x categoria", "chave": ["mes", "categoria"], "dono": "Produto (exemplo)",
    "atualizacao": "reconstruída a cada execução", "pii": False,
    "descricao": "Satisfação por categoria e mês.",
    "colunas": {"mes": "Mês da avaliação (yyyy-MM)", "categoria": "Categoria ('nao_classificado' se o produto está em quarentena)",
                "qtd_avaliacoes": "Número de avaliações", "nota_media": "Média das notas (1 a 5)",
                "pct_notas_baixas": "% de notas 1 ou 2"}},
}
pd.DataFrame([{"tabela": t, "grao": c["grao"], "dono": c["dono"], "pii": c["pii"]} for t, c in CONTRATOS.items()])

,tabela,grao,dono,pii
0,vendas_diarias,data x categoria x estado x canal_aquisicao,Comercial (exemplo),False
1,clientes_360,um cliente,CRM (exemplo),True
2,avaliacoes_resumo,mês x categoria,Produto (exemplo),False


## 6 · Gates: a Gold só é publicada se passar
Três tipos de gate: **grão/chaves**, **governança** (toda coluna descrita, sem PII bruta) e **reconciliação** (receita recalculada por outro caminho, direto da Silver).

In [8]:
PII_PROIBIDA = {"nome", "email", "cpf", "telefone"}

def gates_tabela(nome, df):
    c = CONTRATOS[nome]; ch = c["chave"]; n = df.count()
    regras = [("todas as colunas têm descrição no contrato", set(df.columns) <= set(c["colunas"])),
              ("nenhuma coluna de PII bruta", not (PII_PROIBIDA & set(df.columns))),
              (f"grão: 1 linha por {' x '.join(ch)}", df.select(*ch).distinct().count() == n),
              ("chave do grão sem nulos", df.filter(" OR ".join(f"{k} IS NULL" for k in ch)).count() == 0)]
    if nome == "vendas_diarias":
        regras.append(("receita >= 0", df.filter("receita < 0").count() == 0))
    if nome == "clientes_360":
        regras.append(("todos os clientes da Silver estão aqui", n == spark.table("lake.silver.clientes").count()))
    if nome == "avaliacoes_resumo":
        regras.append(("nota_media entre 1 e 5", df.filter("nota_media < 1 OR nota_media > 5").count() == 0))
    return [{"tabela": nome, "regra": r, "status": "OK" if ok else "ERRO"} for r, ok in regras]

def reconciliar(vd):
    # outro caminho: direto de silver.itens_pedido + silver.pedidos, sem passar por produtos/clientes
    validos = spark.table("lake.silver.pedidos").filter("status <> 'cancelado'").select("pedido_id")
    esp = spark.table("lake.silver.itens_pedido").join(validos, "pedido_id") \
               .agg(F.round(F.sum("valor_item"), 2).alias("r"), F.sum("qtd").alias("q")).first()
    got = vd.agg(F.round(F.sum("receita"), 2).alias("r"), F.sum("qtd_itens").alias("q")).first()
    return [{"tabela": "vendas_diarias", "regra": f"receita Gold ({got.r:,.2f}) = receita Silver ({esp.r:,.2f})",
             "status": "OK" if abs(got.r - esp.r) < 0.01 else "ERRO"},
            {"tabela": "vendas_diarias", "regra": f"itens Gold ({got.q:,}) = itens Silver ({esp.q:,})",
             "status": "OK" if got.q == esp.q else "ERRO"}]

def rodar_gates(gold):
    linhas = sum((gates_tabela(t, df) for t, df in gold.items()), []) + reconciliar(gold["vendas_diarias"])
    return pd.DataFrame(linhas)

### 6.1 · Gates sobre a Gold do `03_gold` (como estava)

In [9]:
gold_v1 = {t: spark.table(f"lake.gold.{t}") for t in CONTRATOS}
g1 = rodar_gates(gold_v1)
print(g1["status"].value_counts().to_string())
g1

status
OK      12
ERRO     5


,tabela,regra,status
0,vendas_diarias,todas as colunas têm descrição no contrato,OK
1,vendas_diarias,nenhuma coluna de PII bruta,OK
2,vendas_diarias,grão: 1 linha por data x categoria x estado x ...,OK
3,vendas_diarias,chave do grão sem nulos,OK
4,vendas_diarias,receita >= 0,OK
5,clientes_360,todas as colunas têm descrição no contrato,ERRO
6,clientes_360,nenhuma coluna de PII bruta,ERRO
7,clientes_360,grão: 1 linha por cliente_id,OK
8,clientes_360,chave do grão sem nulos,OK
9,clientes_360,todos os clientes da Silver estão aqui,OK


In [10]:
# Diagnóstico: de onde vem a diferença de receita?
validos = spark.table("lake.silver.pedidos").filter("status <> 'cancelado'").select("pedido_id")
it_val = spark.table("lake.silver.itens_pedido").join(validos, "pedido_id")
fora = it_val.join(spark.table("lake.silver.produtos").select("produto_id"), "produto_id", "left_anti")
fora.groupBy("produto_id").agg(F.count("*").alias("itens"), F.sum("qtd").alias("qtd"),
                               F.round(F.sum("valor_item"), 2).alias("receita")).orderBy("produto_id").show()
print("produtos na quarentena:", [r.registro for r in spark.table("lake.silver.quarentena").filter("origem = 'produtos'").collect()])
print("linhas de avaliacoes_resumo com categoria NULL:", gold_v1["avaliacoes_resumo"].filter("categoria IS NULL").count())

+----------+-----+---+---------+
|produto_id|itens|qtd|  receita|
+----------+-----+---+---------+
|        55|  210|407|170423.11|
|       114|  222|430| 198664.3|
|       173|  220|445| 137149.0|
+----------+-----+---+---------+



produtos na quarentena: ['{"produto_id":55,"preco":-418.73,"custo":328.22}', '{"produto_id":114,"preco":-462.01,"custo":345.79}', '{"produto_id":173,"preco":-308.2,"custo":231.93}']


linhas de avaliacoes_resumo com categoria NULL: 17


**O gate pegou um erro real.** Os 3 produtos com preço de catálogo negativo foram para a quarentena na Silver, mas os **itens vendidos** desses produtos continuaram em `silver.itens_pedido` (o preço do item no pedido é válido). Na Gold, o `inner join` com `silver.produtos` descartou esses itens **sem avisar**: a receita ficou menor que a da Silver. Uma checagem de nulos não pegaria isso: não há nenhum nulo, a linha simplesmente some.

Decisão: a venda aconteceu, então a receita **fica** na Gold, com `categoria = 'nao_classificado'` (left join) até o cadastro do produto ser corrigido. O mesmo vale para as avaliações desses produtos em `avaliacoes_resumo`. As camadas Silver e Bronze não mudam.

### 6.2 · Gold corrigida (nome mascarado, sem e-mail, left join nos produtos)

In [11]:
NC = F.lit("nao_classificado")

def build_vendas_diarias_v2():
    ped = spark.table("lake.silver.pedidos").filter("status NOT IN ('cancelado')")
    it = spark.table("lake.silver.itens_pedido")
    prod = spark.table("lake.silver.produtos").select("produto_id", "categoria")
    cli = spark.table("lake.silver.clientes").select("cliente_id", "estado", "canal_aquisicao")
    return (it.join(ped.select("pedido_id", "cliente_id", "data_pedido"), "pedido_id")
              .join(prod, "produto_id", "left").join(cli, "cliente_id")
              .withColumn("categoria", F.coalesce("categoria", NC))
              .groupBy(F.to_date("data_pedido").alias("data"), "categoria", "estado", "canal_aquisicao")
              .agg(F.countDistinct("pedido_id").alias("qtd_pedidos"), F.sum("qtd").alias("qtd_itens"),
                   F.round(F.sum("valor_item"), 2).alias("receita")))

MASCARA = "array_join(transform(split(nome, ' '), p -> concat(substr(p, 1, 1), repeat('*', length(p) - 1))), ' ')"

def build_clientes_360_v2():
    df = M.build_clientes_360(spark)               # mesma regra de segmento do 03_gold
    return (df.withColumn("nome_mascarado", F.expr(MASCARA)).drop("nome")
              .select("cliente_id", "nome_mascarado", *[c for c in df.columns if c not in ("cliente_id", "nome")]))

def build_avaliacoes_resumo_v2():
    av = spark.table("lake.silver.avaliacoes")
    pr = spark.table("lake.silver.produtos").select("produto_id", "categoria")
    return (av.join(pr, "produto_id", "left").withColumn("categoria", F.coalesce("categoria", NC))
              .groupBy(F.date_format("data_avaliacao", "yyyy-MM").alias("mes"), "categoria")
              .agg(F.count("*").alias("qtd_avaliacoes"), F.round(F.avg("nota"), 2).alias("nota_media"),
                   F.round(100 * F.avg((F.col("nota") <= 2).cast("int")), 1).alias("pct_notas_baixas")))

gold = {"vendas_diarias": build_vendas_diarias_v2().localCheckpoint(),
        "clientes_360": build_clientes_360_v2().localCheckpoint(),
        "avaliacoes_resumo": build_avaliacoes_resumo_v2().localCheckpoint()}
for t, df in gold.items():
    print(f"{t:<18}{df.count():>7,} linhas")
gold["clientes_360"].select("cliente_id", "nome_mascarado", "estado", "qtd_pedidos", "receita_total", "segmento").show(5)

vendas_diarias     30,755 linhas
clientes_360        5,000 linhas


avaliacoes_resumo     112 linhas


+----------+----------------+------+-----------+-------------+--------+
|cliente_id|  nome_mascarado|estado|qtd_pedidos|receita_total|segmento|
+----------+----------------+------+-----------+-------------+--------+
|         1|   A**** P******|    BA|         12|     10538.23| inativo|
|         2|      A** S*****|    PE|          5|       6298.1| campeao|
|         3|A**** N*********|    CE|          4|      6187.66|em_risco|
|         4|  B**** F*******|    AL|         10|     15023.65| campeao|
|         5|C**** C*********|    BA|          1|       848.99|   ativo|
+----------+----------------+------+-----------+-------------+--------+
only showing top 5 rows



In [12]:
gates = rodar_gates(gold)
print(gates["status"].value_counts().to_string())
gates

status
OK    17


,tabela,regra,status
0,vendas_diarias,todas as colunas têm descrição no contrato,OK
1,vendas_diarias,nenhuma coluna de PII bruta,OK
2,vendas_diarias,grão: 1 linha por data x categoria x estado x ...,OK
3,vendas_diarias,chave do grão sem nulos,OK
4,vendas_diarias,receita >= 0,OK
5,clientes_360,todas as colunas têm descrição no contrato,OK
6,clientes_360,nenhuma coluna de PII bruta,OK
7,clientes_360,grão: 1 linha por cliente_id,OK
8,clientes_360,chave do grão sem nulos,OK
9,clientes_360,todos os clientes da Silver estão aqui,OK


**Prova de que o gate bloqueia:** uma cópia estragada (linha repetida em `vendas_diarias`).

In [13]:
ruim = gold["vendas_diarias"].unionByName(gold["vendas_diarias"].limit(1))
r = pd.DataFrame(gates_tabela("vendas_diarias", ruim))
print(r[r["status"] == "ERRO"].to_string(index=False))
print("-> publicação BLOQUEADA" if (r["status"] == "ERRO").any() else "-> publicação liberada")

        tabela                                                         regra status
vendas_diarias grão: 1 linha por data x categoria x estado x canal_aquisicao   ERRO
-> publicação BLOQUEADA


## 7 · Publicar: Iceberg (fonte da verdade) + contrato no bucket
Só chega aqui se **todos** os gates estiverem OK.

In [14]:
if (gates["status"] == "ERRO").any():
    raise RuntimeError("Gates falharam: a Gold NÃO foi publicada.\n" + gates[gates["status"] == "ERRO"].to_string(index=False))

for t, df in gold.items():
    df.writeTo(f"lake.gold.{t}").using("iceberg").createOrReplace()
    print(f"publicado lake.gold.{t:<18} {spark.table(f'lake.gold.{t}').count():>7,} linhas")

CHAVE_CONTRATO = "lakehouse/gold/_contratos/contratos.json"
s3.put_object(Bucket="warehouse", Key=CHAVE_CONTRATO,
              Body=json.dumps(CONTRATOS, ensure_ascii=False, indent=2).encode("utf-8"))
print(f"\ncontrato publicado em s3://warehouse/{CHAVE_CONTRATO}")
print(json.dumps(json.loads(s3.get_object(Bucket="warehouse", Key=CHAVE_CONTRATO)["Body"].read())["clientes_360"],
                 ensure_ascii=False, indent=2)[:1500])

publicado lake.gold.vendas_diarias      30,755 linhas


publicado lake.gold.clientes_360         5,000 linhas


publicado lake.gold.avaliacoes_resumo      112 linhas

contrato publicado em s3://warehouse/lakehouse/gold/_contratos/contratos.json
{
  "grao": "um cliente",
  "chave": [
    "cliente_id"
  ],
  "dono": "CRM (exemplo)",
  "atualizacao": "reconstruída a cada execução",
  "pii": true,
  "descricao": "Visão do cliente. Nome mascarado; e-mail não entra na Gold (minimização).",
  "colunas": {
    "cliente_id": "Identificador do cliente (pseudônimo)",
    "nome_mascarado": "Nome com letras ocultas (A** S****)",
    "estado": "UF",
    "canal_aquisicao": "Canal de aquisição",
    "data_cadastro": "Data de cadastro",
    "qtd_pedidos": "Pedidos não cancelados",
    "receita_total": "Receita do cliente (R$)",
    "ticket_medio": "Média por pedido (R$)",
    "ultima_compra": "Data da última compra",
    "nota_media": "Média das notas que deu",
    "recencia_dias": "Dias desde a última compra até o último dia da base",
    "segmento": "sem_compra | inativo | campeao | ativo | em_risco"
  }
}


## 8 · Servir: Postgres (cópia) com usuário somente leitura para o Metabase
- A carga usa `truncate=true`: a tabela **não é recriada**, então os painéis e as permissões continuam valendo.
- `clientes_360` mudou de esquema (`nome` → `nome_mascarado`), então ela é recriada **uma única vez** aqui; nas próximas cargas volta o TRUNCATE.
- `metabase_ro` só tem `USAGE` no schema e `SELECT` nas tabelas. `ALTER DEFAULT PRIVILEGES` garante o mesmo para tabelas futuras.

In [15]:
RO_USER = "metabase_ro"
RO_SENHA = os.environ.get("METABASE_PG_PASSWORD", "leitor123")   # laboratório; em produção, cofre de segredos

def pg_exec(sql, user=None, password=None):
    import psycopg2
    cfg = dict(L.PG, **({"user": user, "password": password} if user else {}))
    with psycopg2.connect(**cfg) as c, c.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall() if cur.description else None

colunas_pg = [r[0] for r in pg_exec("SELECT column_name FROM information_schema.columns "
                                    "WHERE table_schema='gold' AND table_name='clientes_360'")]
if "nome" in colunas_pg:                     # migração de esquema, só uma vez
    pg_exec("DROP TABLE gold.clientes_360")
    print("gold.clientes_360 recriada (coluna nome removida, nome_mascarado adicionada)")

for t in gold:
    print("publicado:", L.publish_to_postgres(spark.table(f"lake.gold.{t}"), t, schema="gold"))

pg_exec(f"""DO $$ BEGIN
  IF NOT EXISTS (SELECT FROM pg_roles WHERE rolname = '{RO_USER}') THEN
    CREATE ROLE {RO_USER} LOGIN PASSWORD '{RO_SENHA}';
  END IF; END $$;
  GRANT CONNECT ON DATABASE serving TO {RO_USER};
  GRANT USAGE ON SCHEMA gold TO {RO_USER};
  GRANT SELECT ON ALL TABLES IN SCHEMA gold TO {RO_USER};
  ALTER DEFAULT PRIVILEGES IN SCHEMA gold GRANT SELECT ON TABLES TO {RO_USER};""")

print()
for t in gold:
    n = pg_exec(f"SELECT count(*) FROM gold.{t}")[0][0]
    print(f"gold.{t:<18}{n:>7,} linhas no Postgres | Iceberg: {spark.table(f'lake.gold.{t}').count():,}")
print("\nprivilégios de", RO_USER, ":",
      pg_exec(f"SELECT table_name, string_agg(privilege_type, ',') FROM information_schema.role_table_grants "
              f"WHERE grantee = '{RO_USER}' GROUP BY 1 ORDER BY 1"))

gold.clientes_360 recriada (coluna nome removida, nome_mascarado adicionada)


publicado: gold.vendas_diarias


publicado: gold.clientes_360


publicado: gold.avaliacoes_resumo



gold.vendas_diarias     30,755 linhas no Postgres | Iceberg: 30,755


gold.clientes_360        5,000 linhas no Postgres | Iceberg: 5,000
gold.avaliacoes_resumo     112 linhas no Postgres | Iceberg: 112



privilégios de metabase_ro : [('avaliacoes_resumo', 'SELECT'), ('clientes_360', 'SELECT'), ('vendas_diarias', 'SELECT')]


### 8.1 · Teste do menor privilégio: `metabase_ro` lê, mas não altera

In [16]:
testes = [("SELECT", "SELECT count(*) FROM gold.vendas_diarias"),
          ("INSERT", "INSERT INTO gold.avaliacoes_resumo (mes) VALUES ('2099-01')"),
          ("DELETE", "DELETE FROM gold.vendas_diarias"),
          ("TRUNCATE", "TRUNCATE gold.clientes_360"),
          ("CREATE TABLE", "CREATE TABLE gold.teste (x int)"),
          ("DROP TABLE", "DROP TABLE gold.vendas_diarias")]
for nome, sql in testes:
    try:
        res = pg_exec(sql, RO_USER, RO_SENHA)
        print(f"{nome:<13} PERMITIDO  {res}")
    except Exception as e:
        print(f"{nome:<13} NEGADO     {str(e).splitlines()[0]}")

SELECT        PERMITIDO  [(30755,)]
INSERT        NEGADO     permission denied for table avaliacoes_resumo
DELETE        NEGADO     permission denied for table vendas_diarias
TRUNCATE      NEGADO     permission denied for table clientes_360
CREATE TABLE  NEGADO     permission denied for schema gold
DROP TABLE    NEGADO     must be owner of table vendas_diarias


### 8.2 · Pergunta de negócio (a mesma do Metabase): receita por dia, lida como `metabase_ro`

In [17]:
import psycopg2
with psycopg2.connect(**dict(L.PG, user=RO_USER, password=RO_SENHA)) as c:
    receita_dia = pd.read_sql_query("""
        SELECT data, sum(qtd_pedidos) AS pedidos, round(sum(receita)::numeric, 2) AS receita
        FROM gold.vendas_diarias GROUP BY data ORDER BY data""", c)
    por_mes = pd.read_sql_query("""
        SELECT to_char(data, 'YYYY-MM') AS mes, round(sum(receita)::numeric, 2) AS receita
        FROM gold.vendas_diarias GROUP BY 1 ORDER BY 1""", c)
print(f"{len(receita_dia)} dias | receita total R$ {receita_dia['receita'].sum():,.2f}")
print(receita_dia.tail(7).to_string(index=False))
print()
print(por_mes.to_string(index=False))

541 dias | receita total R$ 40,817,199.93
      data  pedidos   receita
2026-06-24    152.0 169971.58
2026-06-25    142.0 151209.24
2026-06-26    131.0 145466.19
2026-06-27    148.0 156120.15
2026-06-28    131.0 139731.50
2026-06-29    149.0 143598.79
2026-06-30    133.0 137275.47

    mes    receita
2025-01  150828.98
2025-02  393820.72
2025-03  657728.31
2025-04  967768.09
2025-05 1274995.94
2025-06 1399788.02
2025-07 1746221.77
2025-08 1942708.79
2025-09 2083611.01
2025-10 2582563.53
2025-11 2710651.60
2025-12 2847829.89
2026-01 3240604.44
2026-02 3176262.76
2026-03 3600335.45
2026-04 3679051.96
2026-05 4046811.21
2026-06 4315617.46


/tmp/ipykernel_2302718/1402585660.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  receita_dia = pd.read_sql_query("""
/tmp/ipykernel_2302718/1402585660.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  por_mes = pd.read_sql_query("""


## Checkpoint
- [x] Inventário de landing e warehouse e contagem de linhas por tabela.
- [x] DQ com histórico (append = snapshot novo por execução).
- [x] Versionamento do bucket: erro recuperado pelo MinIO; versões antigas só somem apagando cada `VersionId`.
- [x] Gates: falharam na Gold original (reconciliação), passaram na Gold corrigida; publicação bloqueada quando há ERRO.
- [x] Contrato publicado em `s3://warehouse/lakehouse/gold/_contratos/contratos.json`.
- [x] Postgres com as 3 tabelas e `metabase_ro` só com SELECT.